In [19]:
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF


# Locate Module 15 data without exposing any personal file paths.
def find_data_folder():
    cwd = Path.cwd()
    candidates = []
    for parent in (cwd, *cwd.parents):
        candidates.extend([
            parent / "Data",
            parent / "4. Module 15" / "Data",
            parent / "7. Capstone Project" / "4. Module 15" / "Data",
        ])
    for candidate in candidates:
        if all((candidate / f"function_{i}.csv").is_file() for i in range(1, 9)):
            return candidate
    raise FileNotFoundError(
        "Could not locate all eight function CSV files in the Module 15 Data folder. "
        "Open Jupyter from the capstone project folder or Module 15 folder."
    )

folder = find_data_folder()


In [20]:
data = {}
for i in range(1, 9):
    data[i] = pd.read_csv(folder / f"function_{i}.csv")
print("All eight datasets loaded successfully.")


All eight datasets loaded successfully.


In [21]:
# Dataset dimensions and observation counts
from IPython.display import display
display(pd.DataFrame([{"Function": i, "Observations": len(data[i]), "Dimensions": len(data[i].columns)-1} for i in range(1,9)]).set_index("Function"))

,Observations,Dimensions
Function,,
1,13,2
2,13,2
3,18,3
4,33,4
5,23,4
6,23,5
7,33,6
8,43,8


In [22]:
# Points submitted last week
new_inputs = {
    1: [0.000045, 0.003062],
    2: [0.709627, 0.881953],
    3: [0.732035, 0.997300, 0.015653],
    4: [0.391976, 0.424757, 0.494823, 0.464424],
    5: [0.671629, 0.981118, 0.990458, 0.985189],
    6: [0.013758, 0.147930, 0.445597, 0.994845, 0.013469],
    7: [0.021549, 0.313334, 0.360216, 0.141732, 0.295328, 0.686548],
    8: [0.025529, 0.451238, 0.098209, 0.031391,
        0.412421, 0.138573, 0.114934, 0.994361]
}

# Outputs returned this week
new_outputs = {
    1: 1.909096326648326e-246,
    2: 0.5486083534111039,
    3: -0.11514149202782288,
    4: -1.850268223115655,
    5: 4595.479318470633,
    6: -1.182842192025624,
    7: 2.1395513389520344,
    8: 9.6453009842564
}

In [23]:
# Add each returned observation only if it is not already present.
new_observation_indices = {}
update_rows = []
for i in range(1, 9):
    input_columns = [col for col in data[i].columns if col != "output"]
    x_new = np.asarray(new_inputs[i], dtype=float)
    existing_x = data[i][input_columns].to_numpy(dtype=float)
    matching = np.flatnonzero(np.all(np.isclose(existing_x, x_new, atol=5e-7, rtol=0), axis=1))
    if len(matching):
        idx = int(matching[0])
        existing_y = float(data[i].iloc[idx]["output"])
        if not np.isclose(existing_y, new_outputs[i], rtol=1e-8, atol=1e-12):
            raise ValueError(f"Function {i}: existing observation has a different output.")
        status = "Already present"
    else:
        new_row = dict(zip(input_columns, x_new))
        new_row["output"] = new_outputs[i]
        data[i] = pd.concat([data[i], pd.DataFrame([new_row])], ignore_index=True)
        idx = len(data[i]) - 1
        status = "Added"
    new_observation_indices[i] = idx
    update_rows.append({"Function":i, "Status":status, "Observations":len(data[i])})
display(pd.DataFrame(update_rows).set_index("Function"))

,Status,Observations
Function,,
1,Already present,13
2,Already present,13
3,Already present,18
4,Already present,33
5,Already present,23
6,Already present,23
7,Already present,33
8,Already present,43


In [24]:
# Compare new outputs against the best of all other observations.
performance_rows = []
for i in range(1,9):
    previous_data = data[i].drop(index=new_observation_indices[i])
    previous_best = previous_data["output"].max()
    new_output = new_outputs[i]
    updated_best = data[i]["output"].max()
    performance_rows.append({"Function":i,"Previous best":previous_best,"New output":new_output,
                             "Difference":new_output-previous_best,"Updated best":updated_best,
                             "Result":"New best" if new_output>previous_best else "No improvement"})
display(pd.DataFrame(performance_rows).set_index("Function").round(6))

,Previous best,New output,Difference,Updated best,Result
Function,,,,,
1,0.000000,0.000000,-0.000000,0.000000,No improvement
2,0.611205,0.548608,-0.062597,0.611205,No improvement
3,-0.020167,-0.115141,-0.094975,-0.020167,No improvement
4,-0.300645,-1.850268,-1.549624,-0.300645,No improvement
5,2765.819787,4595.479318,1829.659532,4595.479318,New best
6,-0.458511,-1.182842,-0.724332,-0.458511,No improvement
7,1.950612,2.139551,0.188939,2.139551,New best
8,9.821986,9.645301,-0.176685,9.821986,No improvement


In [25]:
# Refit Gaussian processes with the original 10 optimiser restarts.
models_week4 = {}
fit_rows = []
for i in range(1,9):
    X = data[i].drop(columns="output").values
    y = data[i]["output"].values
    kernel = RBF(length_scale=1.0, length_scale_bounds=(1e-5,1e5))
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True,
                                  n_restarts_optimizer=10, random_state=42)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        gp.fit(X,y)
    models_week4[i]=gp
    fit_rows.append({"Function":i,"Fitted RBF length scale":gp.kernel_.length_scale,
                     "Convergence warnings":sum(issubclass(w.category,ConvergenceWarning) for w in caught)})
display(pd.DataFrame(fit_rows).set_index("Function"))

,Fitted RBF length scale,Convergence warnings
Function,,
1,0.000010,1
2,0.051605,0
3,0.197676,0
4,0.428258,0
5,0.366869,0
6,0.000010,1
7,0.295718,0
8,0.948993,0


In [26]:
# Compare GP predictions and uncertainty for Functions 1 and 6.
rng = np.random.default_rng(42)
diagnostic_rows=[]
for i in [1,6]:
    X=data[i].drop(columns="output").values
    X_candidates=rng.uniform(0,1,size=(100000,X.shape[1]))
    mean,std=models_week4[i].predict(X_candidates,return_std=True)
    diagnostic_rows.append({"Function":i,"Mean min":mean.min(),"Mean max":mean.max(),
                            "Mean range":mean.max()-mean.min(),"Std min":std.min(),
                            "Std max":std.max(),"Std range":std.max()-std.min()})
display(pd.DataFrame(diagnostic_rows).set_index("Function"))

,Mean min,Mean max,Mean range,Std min,Std max,Std range
Function,,,,,,
1,-0.000277,-1.538440e-08,0.000277,0.000010,0.000961,0.000951
6,-1.487422,-1.487422e+00,0.000000,0.534192,0.534192,0.000000


In [27]:
# Compare kappa = 1.5, 2.5 and 3.5

kappa_values = [1.5, 2.5, 3.5]
kappa_functions = [2, 3, 4, 5, 7, 8]

n_candidates = 100000

kappa_results = {}
kappa_rows = []

for i in kappa_functions:

    X_existing = data[i].drop(columns="output").values
    y_existing = data[i]["output"].values

    gp = models_week4[i]
    n_dimensions = X_existing.shape[1]

    # Use the same candidate set for all kappa values
    rng = np.random.default_rng(42 + i)

    X_candidates = rng.uniform(
        0,
        1,
        size=(n_candidates, n_dimensions)
    )

    # GP predictions only need to be calculated once
    mean, std = gp.predict(
        X_candidates,
        return_std=True
    )

    # Current best observation
    best_index = np.argmax(y_existing)
    current_best_point = X_existing[best_index]
    current_best_output = y_existing[best_index]

    kappa_results[i] = {}

    for kappa in kappa_values:

        ucb = mean + kappa * std
        selected_index = np.argmax(ucb)

        point = X_candidates[selected_index]

        distance_best = np.linalg.norm(
            point - current_best_point
        )

        distances_existing = np.linalg.norm(
            X_existing - point,
            axis=1
        )

        nearest_distance = distances_existing.min()

        kappa_results[i][kappa] = {
            "point": point,
            "predicted_mean": mean[selected_index],
            "uncertainty": std[selected_index],
            "ucb": ucb[selected_index],
            "distance_best": distance_best,
            "nearest_distance": nearest_distance
        }

        kappa_rows.append({"Function":i,"κ":kappa,"Predicted output":mean[selected_index],
                           "Uncertainty":std[selected_index],"UCB":ucb[selected_index],
                           "Distance from best":distance_best,"Nearest observation":nearest_distance,
                           "Selected coordinates":np.array2string(point,precision=6,separator=", ")})

display(pd.DataFrame(kappa_rows).set_index(["Function","κ"]).round(6))

Predicted output  Uncertainty          UCB  Distance from best  \
Function κ                                                                     
2        1.5          0.556031     0.121843     0.738795            0.038662   
         2.5          0.375397     0.210671     0.902075            0.863646   
         3.5          0.342377     0.221941     1.119170            0.756599   
3        1.5         -0.033932     0.053728     0.046659            0.612631   
         2.5         -0.075090     0.074427     0.110977            0.361547   
         3.5         -0.082998     0.077189     0.187163            0.415883   
4        1.5         -0.309086     0.456642     0.375877            0.091845   
         2.5         -0.694545     0.674163     0.990863            0.135978   
         3.5        -18.361781     6.681320     5.022841            1.010882   
5        1.5       4561.793859   452.311632  5240.261307            0.193062   
         2.5       4090.674685   726.333454  5906.508319            0.329141   
         3.5       4090.674685   726.333454  6632.841773            0.329141   
7        1.5          1.959149     0.276601     2.374051            0.176210   
         2.5          1.959149     0.276601     2.650652            0.176210   
         3.5          1.641296     0.383505     2.983562            0.301962   
8        1.5         10.059119     0.187332    10.340117            0.475599   
         2.5          9.551346     0.451204    10.679357            0.882422   
         3.5          9.253277     0.569759    11.247434            1.082323   

              Nearest observation  \
Function κ                          
2        1.5             0.038662   
         2.5             0.065205   
         3.5             0.075892   
3        1.5             0.192104   
         2.5             0.303182   
         3.5             0.354122   
4        1.5             0.091845   
         2.5             0.135978   
         3.5             0.566853   
5        1.5             0.193062   
         2.5             0.329141   
         3.5             0.329141   
7        1.5             0.176210   
         2.5             0.176210   
         3.5             0.291121   
8        1.5             0.475599   
         2.5             0.833699   
         3.5             0.879561   

                                           Selected coordinates  
Function κ                                                       
2        1.5                               [0.73856 , 0.912272]  
         2.5                               [0.688986, 0.063026]  
         3.5                               [0.725891, 0.170323]  
3        1.5                     [0.41795 , 0.732195, 0.005457]  
         2.5                     [0.511158, 0.731701, 0.617932]  
         3.5                     [0.565168, 0.760792, 0.653023]  
4        1.5           [0.44374 , 0.493644, 0.40979 , 0.452219]  
         2.5           [0.476265, 0.525427, 0.417514, 0.439754]  
         3.5           [0.047215, 0.978118, 0.999658, 0.02652 ]  
5        1.5           [0.851064, 0.985689, 0.919598, 0.97933 ]  
         2.5           [0.994129, 0.987119, 0.927372, 0.967532]  
         3.5           [0.994129, 0.987119, 0.927372, 0.967532]  
7        1.5  [0.091435, 0.306185, 0.316439, 0.020166, 0.225...  
         2.5  [0.091435, 0.306185, 0.316439, 0.020166, 0.225...  
         3.5  [0.178516, 0.448011, 0.378124, 0.00201 , 0.138...  
8        1.5  [0.137618, 0.223807, 0.054101, 0.335736, 0.528...  
         2.5  [0.410732, 0.64041 , 0.007318, 0.44934 , 0.728...  
         3.5  [0.356959, 0.085948, 0.02191 , 0.938974, 0.856...

In [28]:
# Compare how far the selected points move as κ changes.
distance_rows=[]
for i in kappa_functions:
    p15=kappa_results[i][1.5]["point"]
    p25=kappa_results[i][2.5]["point"]
    p35=kappa_results[i][3.5]["point"]
    distance_rows.append({"Function":i,"1.5 → 2.5":np.linalg.norm(p15-p25),
                          "1.5 → 3.5":np.linalg.norm(p15-p35),"2.5 → 3.5":np.linalg.norm(p25-p35)})
display(pd.DataFrame(distance_rows).set_index("Function").round(4))

,1.5 → 2.5,1.5 → 3.5,2.5 → 3.5
Function,,,
2,0.8507,0.7421,0.1135
3,0.6195,0.6647,0.0707
4,0.0478,0.9597,0.9480
5,0.1438,0.1438,0.0000
7,0.0000,0.1986,0.1986
8,0.7627,0.9229,0.9939


In [29]:
from scipy.stats import norm

# Expected Improvement acquisition function
def expected_improvement(mean, std, best_y, xi=0.01):

    improvement = mean - best_y - xi

    with np.errstate(divide="ignore", invalid="ignore"):
        Z = improvement / std

        ei = (
            improvement * norm.cdf(Z)
            + std * norm.pdf(Z)
        )

    # EI should be zero where uncertainty is zero
    ei[std == 0.0] = 0.0

    return ei

In [30]:
# Compare Expected Improvement with provisional UCB selections

ei_results = {}
ei_rows = []

for i in kappa_functions:

    X_existing = data[i].drop(columns="output").values
    y_existing = data[i]["output"].values

    gp = models_week4[i]
    n_dimensions = X_existing.shape[1]

    # Recreate the same candidate set used in Step 3
    rng = np.random.default_rng(42 + i)

    X_candidates = rng.uniform(
        0,
        1,
        size=(100000, n_dimensions)
    )

    mean, std = gp.predict(
        X_candidates,
        return_std=True
    )

    # Current best
    best_index = np.argmax(y_existing)
    current_best_point = X_existing[best_index]
    current_best_output = y_existing[best_index]

    # Calculate Expected Improvement
    ei = expected_improvement(
        mean,
        std,
        current_best_output,
        xi=0.01
    )

    selected_index = np.argmax(ei)
    point = X_candidates[selected_index]

    # Distances
    distance_best = np.linalg.norm(
        point - current_best_point
    )

    nearest_distance = np.linalg.norm(
        X_existing - point,
        axis=1
    ).min()

    # Compare with provisional kappa = 1.5 point
    ucb_point = kappa_results[i][1.5]["point"]

    distance_from_ucb = np.linalg.norm(
        point - ucb_point
    )

    ei_results[i] = {
        "point": point,
        "predicted_mean": mean[selected_index],
        "uncertainty": std[selected_index],
        "ei": ei[selected_index],
        "distance_best": distance_best,
        "nearest_distance": nearest_distance,
        "distance_from_ucb": distance_from_ucb
    }

    ei_rows.append({"Function":i,"Predicted output":mean[selected_index],
                    "Uncertainty":std[selected_index],"Expected improvement":ei[selected_index],
                    "Distance from best":distance_best,"Nearest observation":nearest_distance,
                    "Distance from UCB":distance_from_ucb,
                    "Selected coordinates":np.array2string(point,precision=6,separator=", ")})

display(pd.DataFrame(ei_rows).set_index("Function").round(6))

,Predicted output,Uncertainty,Expected improvement,Distance from best,Nearest observation,Distance from UCB,Selected coordinates
Function,,,,,,,
2,0.590106,0.095042,0.024378,0.030161,0.030161,0.008503,"[0.730707, 0.915532]"
3,-0.031179,0.051758,0.011821,0.608589,0.178778,0.022809,"[0.433866, 0.716516, 0.000863]"
4,-0.309086,0.456642,0.173102,0.091845,0.091845,0.000000,"[0.44374 , 0.493644, 0.40979 , 0.452219]"
5,4561.793859,452.311632,164.098981,0.193062,0.193062,0.000000,"[0.851064, 0.985689, 0.919598, 0.97933 ]"
7,2.053815,0.212603,0.045405,0.149214,0.149214,0.189264,"[0.154752, 0.313081, 0.341855, 0.169387, 0.318..."
8,10.080326,0.149570,0.251350,0.518070,0.518070,0.262091,"[0.14854 , 0.297539, 0.20186 , 0.328776, 0.636..."


In [31]:
from sklearn.metrics import pairwise_distances

alternative_selected = {}
exploration_rows = []

for i in [1, 6]:

    X_existing = data[i].drop(columns="output").values
    n_dimensions = X_existing.shape[1]

    rng = np.random.default_rng(42 + i)

    X_candidates = rng.uniform(
        0,
        1,
        size=(100000, n_dimensions)
    )

    # Distance from each candidate to all existing observations
    distances = pairwise_distances(
        X_candidates,
        X_existing
    )

    # Distance to nearest existing observation
    nearest_distances = distances.min(axis=1)

    # Select candidate furthest from its nearest observation
    best_index = np.argmax(nearest_distances)

    selected_point = X_candidates[best_index]
    max_distance = nearest_distances[best_index]

    alternative_selected[i] = selected_point

    # Current best point
    best_obs_index = data[i]["output"].idxmax()

    current_best_point = (
        data[i]
        .drop(columns="output")
        .loc[best_obs_index]
        .values
        .astype(float)
    )

    distance_best = np.linalg.norm(
        selected_point - current_best_point
    )

    exploration_rows.append({"Function":i,"Nearest observation":max_distance,
                             "Distance from best":distance_best,
                             "Selected coordinates":np.array2string(selected_point,precision=6,separator=", ")})
display(pd.DataFrame(exploration_rows).set_index("Function").round(6))

,Nearest observation,Distance from best,Selected coordinates
Function,,,
1,0.315336,0.413652,"[0.41743 , 0.463246]"
6,0.878156,0.880958,"[0.033216, 0.058439, 0.124012, 0.011504, 0.042..."


In [32]:
# Final Week 4 query selections

final_points = {
    1: alternative_selected[1],
    2: ei_results[2]["point"],
    3: ei_results[3]["point"],
    4: ei_results[4]["point"],
    5: ei_results[5]["point"],
    6: alternative_selected[6],
    7: ei_results[7]["point"],
    8: ei_results[8]["point"]
}

selection_methods = {
    1: "Maximum-distance exploration",
    2: "Expected Improvement",
    3: "Expected Improvement",
    4: "EI / UCB agreement",
    5: "EI / UCB agreement",
    6: "Maximum-distance exploration",
    7: "Expected Improvement",
    8: "Expected Improvement"
}

In [33]:
# Final validation checks

validation_rows = []
for i in range(1, 9):

    point = np.asarray(final_points[i])
    X_existing = data[i].drop(columns="output").values

    # Check permitted range
    in_bounds = np.all(
        (point >= 0) & (point <= 1)
    )

    # Distance from all existing observations
    distances = np.linalg.norm(
        X_existing - point,
        axis=1
    )

    nearest_distance = distances.min()

    # Check for duplicate
    duplicate = np.any(
        np.all(
            np.isclose(
                X_existing,
                point,
                atol=1e-8
            ),
            axis=1
        )
    )

    # Current best point
    best_index = data[i]["output"].idxmax()

    best_point = (
        data[i]
        .drop(columns="output")
        .loc[best_index]
        .values
        .astype(float)
    )

    distance_best = np.linalg.norm(
        point - best_point
    )

    validation_rows.append({"Function":i,"Method":selection_methods[i],"In bounds":bool(in_bounds),
                            "Duplicate":bool(duplicate),"Nearest observation":nearest_distance,
                            "Distance from best":distance_best})
validation_table=pd.DataFrame(validation_rows).set_index("Function")
display(validation_table.round(6))
assert validation_table["In bounds"].all() and not validation_table["Duplicate"].any()

,Method,In bounds,Duplicate,Nearest observation,Distance from best
Function,,,,,
1,Maximum-distance exploration,True,False,0.315336,0.413652
2,Expected Improvement,True,False,0.030161,0.030161
3,Expected Improvement,True,False,0.178778,0.608589
4,EI / UCB agreement,True,False,0.091845,0.091845
5,EI / UCB agreement,True,False,0.193062,0.193062
6,Maximum-distance exploration,True,False,0.878156,0.880958
7,Expected Improvement,True,False,0.149214,0.149214
8,Expected Improvement,True,False,0.518070,0.518070


In [34]:
for i in range(1, 9):
    submission = "-".join(
        f"{value:.6f}"
        for value in final_points[i]
    )
    print(f"Function {i}: {submission}")

Function 1: 0.417430-0.463246
Function 2: 0.730707-0.915532
Function 3: 0.433866-0.716516-0.000863
Function 4: 0.443740-0.493644-0.409790-0.452219
Function 5: 0.851064-0.985689-0.919598-0.979330
Function 6: 0.033216-0.058439-0.124012-0.011504-0.042834
Function 7: 0.154752-0.313081-0.341855-0.169387-0.318877-0.633022
Function 8: 0.148540-0.297539-0.201860-0.328776-0.636564-0.500531-0.197949-0.479426
